# Demo 01 — Learning as Compression

**Class 01 · D1-D3** (slides: `slides/01_foundations.md`)

Three live demos, each with more experiments than the slides show:

* **D1 — Memorization and generalization (Example 1):** a lookup table against a fitted curve, the errors by hand, sweeps over the sample size and the noise.
* **D2 — Entropy and compression:** two coins by hand, the entropy curve, and what `zlib` and `bz2` achieve on seven kinds of data.
* **D3 — Model selection as compression (MDL):** the description length of polynomial degrees, and when it fails.

A short bridge at the end shows that *finding* the model is an optimization problem (Class 02 expands it).

**Example 1** is a noisy sine, $y = \sin(2\pi x) + \varepsilon$ with $n = 50$ and $\sigma = 0.3$.
Every cell that prints a number of a slide starts with `slide: <frame title>`.
Plots are written to `assets/figures/plots/01/` by `plotlib.save` (transparent PDF at the size on the slide).

In [ ]:
import bz2
import math
import sys
import textwrap
import zlib
from collections import Counter
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display
from numpy.polynomial import Polynomial

sys.path.append("..")
import plotlib  # noqa: E402

plotlib.setup(module="01")
BLUE, ORANGE, GREEN, RED, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED, SIGMA = 42, 0.3
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_width_chars(160)


def f(x):
    """The true pattern of Example 1."""
    return np.sin(2 * np.pi * x)


def example1(n=50, sigma=SIGMA, seed=SEED, n_test=1000):
    """Example 1: a noisy sine. Fresh generator per call, so any section can be rerun alone."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    y = f(x) + rng.normal(0, sigma, n)
    x_test = rng.uniform(0, 1, n_test)
    y_test = f(x_test) + rng.normal(0, sigma, n_test)
    return x, y, x_test, y_test


def mse(y, y_hat):
    return float(np.mean((y - y_hat) ** 2))


def lookup_table(x_fit, y_fit):
    """Memorizer: copies the answer of the closest stored x."""

    def predict(x_query):
        return y_fit[np.abs(x_query[:, None] - x_fit[None, :]).argmin(axis=1)]

    return predict


def entropy_bits(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum()) + 0.0


def bits_per_symbol(data: bytes, codec) -> float:
    return 8 * len(codec(data)) / len(data)


def show(table: pl.DataFrame, digits: int = 4) -> None:
    """Print a table with rounded floats."""
    print(table.with_columns(pl.col(pl.Float64).round(digits)))


def slide(title: str) -> None:
    print(f"--- slide: {title} ---")


grid = np.linspace(0, 1, 400)

## D1 — Memorization and generalization

The data come from an unknown function plus noise:

$$y_i = f(x_i) + \varepsilon_i, \qquad f(x) = \sin(2\pi x), \quad \varepsilon_i \sim \mathcal{N}(0, 0.3^2)$$

Two learners: a **lookup table** that stores every training pair (100 numbers), and a **cubic polynomial** with 4 numbers.

### The data (slides: Notation N1: data, Data = Pattern + Noise; plot `d1_data`)

In [ ]:
# slide: Notation N1: data; Data = Pattern + Noise
slide("Notation N1: data; Data = Pattern + Noise")
x_train, y_train, x_test, y_test = example1()
n, d = len(x_train), 1
eps = y_train - f(x_train)
print(
    f"n = {n} training examples, d = {d} feature, {len(x_test)} test examples, sigma = {SIGMA}, sigma^2 = {SIGMA**2:.2f}"
)
show(
    pl.DataFrame(
        {
            "i": [1, 2, 3, 4, 5],
            "x_i": x_train[:5],
            "y_i": y_train[:5],
            "f(x_i)": f(x_train[:5]),
            "eps_i": eps[:5],
        }
    )
)
print(
    f"mean of the {n} noise values = {eps.mean():+.4f}, standard deviation = {eps.std(ddof=1):.4f} (true: 0 and {SIGMA})"
)

In [ ]:
# plot d1_data (frame: Data = Pattern + Noise)
fg, ax = plotlib.fig("full", 1.6)
ax.plot(grid, f(grid), color=GREEN, ls="--", label="pattern $f(x)$")
ax.scatter(x_train, y_train, s=9, color=BLUE, zorder=3, label="data $y_i = f(x_i) + \\varepsilon_i$")
k = int(np.argmax(np.abs(eps)))
ax.plot([x_train[k]] * 2, [f(x_train[k]), y_train[k]], color=RED, lw=1.2, zorder=4)
ax.annotate(f"$\\varepsilon_{{{k + 1}}}$", (x_train[k], (f(x_train[k]) + y_train[k]) / 2), (-3, 0), textcoords="offset points",
            color=RED, fontsize=8, va="center", ha="right")  # fmt: skip
ax.set(xlabel="$x$", ylabel="$y$", xlim=(0, 1))
ax.legend(loc="upper right", ncols=1, handletextpad=0.4, borderaxespad=0.1)
display(fg)
plotlib.save(fg, "d1_data")

### Lookup table against the cubic (slides: Memorization: the Lookup Table, Generalization; plot `d1_lookup_vs_cubic`)

In [ ]:
# slide: Memorization: the Lookup Table; Generalization
slide("Memorization: the Lookup Table; Generalization")
table_model = lookup_table(x_train, y_train)
cubic = Polynomial.fit(x_train, y_train, deg=3)
res = pl.DataFrame(
    {
        "learner": ["lookup table", "cubic model"],
        "train_mse": [mse(y_train, table_model(x_train)), mse(y_train, cubic(x_train))],
        "test_mse": [mse(y_test, table_model(x_test)), mse(y_test, cubic(x_test))],
        "numbers_stored": [2 * n, 4],
    }
)
show(res)
print(
    f"noise floor sigma^2 = {SIGMA**2:.4f}; the lookup table's test MSE / sigma^2 = {res['test_mse'][0] / SIGMA**2:.2f}"
)
print("cubic coefficients (power basis, x in [0, 1]):", np.round(cubic.convert().coef, 3))

In [ ]:
# plot d1_lookup_vs_cubic
fg, axs = plotlib.fig("full", 1.7, ncols=2, sharey=True)
for ax, pred, title, tr, te in [
    (axs[0], table_model(grid), "lookup table (100 numbers)", res["train_mse"][0], res["test_mse"][0]),
    (axs[1], cubic(grid), "cubic (4 numbers)", res["train_mse"][1], res["test_mse"][1]),
]:
    ax.plot(grid, f(grid), color=GREEN, ls="--", lw=1, zorder=1)
    ax.scatter(x_train, y_train, s=6, color=GREY, zorder=2)
    if ax is axs[0]:
        ax.step(grid, pred, where="mid", color=RED, lw=1.1, zorder=3)
    else:
        ax.plot(grid, pred, color=BLUE, lw=1.6, zorder=3)
    ax.set_title(f"{title}\ntrain MSE {tr:.3f}, test MSE {te:.3f}", fontsize=7.5)
    ax.set(xlabel="$x$", xlim=(0, 1))
axs[0].set_ylabel("$y$")
display(fg)
plotlib.save(fg, "d1_lookup_vs_cubic")

### Sum, mean and error by hand (slides: Sum and Mean, Errors and Mean Squared Error; plot `d1_errors`)

The first five training examples of Example 1 and the cubic of the cell above.

In [ ]:
# slide: Sum and Mean; Errors and Mean Squared Error
slide("Sum and Mean; Errors and Mean Squared Error")
toy = np.array([2, 5, 3, 6])
print(
    f"toy values {toy.tolist()}: sum of the first three = {toy[:3].sum()}, sum = {toy.sum()}, mean = {toy.mean():.1f}"
)
y4 = y_train[:4]
print(f"Example 1, y_1..y_4 = {np.round(y4, 4).tolist()}: sum = {y4.sum():.4f}, mean = {y4.mean():.4f} (sum / 4)")
x5, y5 = x_train[:5], y_train[:5]
yh5 = cubic(x5)
e5 = y5 - yh5
tab = pl.DataFrame({"i": [1, 2, 3, 4, 5], "x_i": x5, "y_i": y5, "yhat_i": yh5, "e_i": e5, "e_i^2": e5**2})
show(tab)
print(f"sum of e_i = {e5.sum():+.4f} (positive and negative errors cancel)")
print(f"sum of e_i^2 = {np.sum(e5**2):.4f}; MSE of these 5 = {np.mean(e5**2):.4f}")
print(
    f"MSE of all {n} training examples = {mse(y_train, cubic(x_train)):.4f}; on the {len(x_test)} test examples = {mse(y_test, cubic(x_test)):.4f}"
)
toy_e = np.array([1, -0.5, 0.5])
print(f"toy errors {toy_e.tolist()}: squares {(toy_e**2).tolist()}, sum {np.sum(toy_e**2)}, MSE {np.mean(toy_e**2)}")

In [ ]:
# plot d1_errors
fg, ax = plotlib.fig("full", 1.7)
ax.plot(grid, cubic(grid), color=BLUE, lw=1.4, label="model $\\hat{y}$")
for i in range(5):
    ax.plot([x5[i]] * 2, [yh5[i], y5[i]], color=RED, lw=1.3, zorder=3)
    up = e5[i] > 0
    ax.annotate(f"$e_{{{i + 1}}}$", (x5[i], y5[i]), (0, 3 if up else -3), textcoords="offset points",
                color=RED, fontsize=8, ha="center", va="bottom" if up else "top")  # fmt: skip
ax.scatter(x5, y5, s=14, color="k", zorder=4, label="data $y_i$")
ax.set(xlabel="$x$", ylabel="$y$", xlim=(0, 1), ylim=(-1.3, 1.1))
ax.legend(loc="upper right", borderaxespad=0.1)
display(fg)
plotlib.save(fg, "d1_errors")

### Sweep: when does memorization stop hurting? (no slide)

Mean over 20 seeds. The noise floor is $\sigma^2$: no model can beat it on new data.

In [ ]:
rows = []
for n_s in (10, 50, 500):
    for sigma_s in (0.1, 0.3, 1.0):
        acc = []
        for s in range(20):
            xa, ya, xb, yb = example1(n_s, sigma_s, seed=SEED + s)
            lt, cu = lookup_table(xa, ya), Polynomial.fit(xa, ya, deg=3)
            acc.append([mse(ya, lt(xa)), mse(yb, lt(xb)), mse(ya, cu(xa)), mse(yb, cu(xb))])
        a = np.mean(acc, axis=0)
        rows.append({
            "n": n_s, "sigma": sigma_s, "noise_floor": sigma_s**2, "lookup_train": a[0], "lookup_test": a[1],
            "cubic_train": a[2], "cubic_test": a[3], "better": "lookup" if a[1] < a[3] else "cubic",
        })  # fmt: skip
sweep1 = pl.DataFrame(rows)
show(sweep1)

**Reading the sweep.** The lookup table's test error stays near $2\sigma^2$ for every $n$: it copies the noise of the stored point and adds the noise of the new one.
More data does not cure memorization. The cubic stays near $\sigma^2$ plus its own small bias, except for $n = 10$: four parameters on ten noisy points
still follow the noise, and at $\sigma \ge 0.3$ the lookup table is then no worse (the **failure case** of the cubic).

### A flexible model mistakes noise for signal (slide: Mistaking Noise for Signal; plot `d1_wiggle`)

In [ ]:
# slide: Mistaking Noise for Signal
slide("Mistaking Noise for Signal")
xw, yw, xw_test, yw_test = example1(15, SIGMA, seed=SEED)
rows = []
for q in (3, 9, 14):
    p = Polynomial.fit(xw, yw, deg=q)
    rows.append({"degree": q, "train_mse": mse(yw, p(xw)), "test_mse": mse(yw_test, p(xw_test))})
show(pl.DataFrame(rows))
print("n = 15 training points, noise sigma = 0.3")

In [ ]:
# plot d1_wiggle
fg, ax = plotlib.fig("full", 1.7)
ax.plot(grid, f(grid), color=GREEN, ls="--", label="pattern $f(x)$")
ax.plot(grid, Polynomial.fit(xw, yw, deg=3)(grid), color=BLUE, label="degree 3")
ax.plot(grid, Polynomial.fit(xw, yw, deg=9)(grid), color=RED, label="degree 9")
ax.scatter(xw, yw, s=11, color="k", zorder=4, label="15 points")
ax.set(xlabel="$x$", ylabel="$y$", xlim=(0, 1), ylim=(-2.2, 2.2))
ax.legend(loc="upper right", ncols=2, columnspacing=1.0, handletextpad=0.4, borderaxespad=0.1)
display(fg)
plotlib.save(fg, "d1_wiggle")

## D2 — Entropy and compression

Shannon entropy of a source with outcome probabilities $p(x)$, in bits per outcome:

$$H(X) = -\sum_{x} p(x)\log_2 p(x), \qquad h(x) = -\log_2 p(x) \text{ (the surprise of one outcome)}$$

### Probability (slide: Probability; plot `d2_probability`)

In [ ]:
# slide: Probability
slide("Probability")
rng = np.random.default_rng(SEED)
outcomes, p_true = ["sun", "cloud", "rain", "snow"], np.array([0.5, 0.25, 0.2, 0.05])
days = rng.choice(len(outcomes), size=1000, p=p_true)
freq = np.bincount(days, minlength=len(outcomes)) / len(days)
show(pl.DataFrame({"outcome": outcomes, "p(x)": p_true, "count of 1000 days": np.bincount(days), "frequency": freq}))
print(f"sum of p(x) = {p_true.sum():.2f}; sum of the frequencies = {freq.sum():.2f}")

In [ ]:
# plot d2_probability
fg, ax = plotlib.fig("col", 1.6)
xs = np.arange(len(outcomes))
ax.bar(xs, p_true, color=BLUE, width=0.55, label="$p(x)$")
ax.scatter(xs, freq, color=ORANGE, s=18, zorder=4, label="frequency, 1000 days")
for xi, pi, fi in zip(xs, p_true, freq):
    ax.annotate(f"{pi:g}", (xi, max(pi, fi)), (0, 5), textcoords="offset points", ha="center", fontsize=7.5)
ax.set(xticks=xs, xticklabels=outcomes, ylabel="$p(x)$", ylim=(0, 0.62))
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2, borderaxespad=0.1, columnspacing=0.8, handletextpad=0.3
)
ax.grid(axis="x", visible=False)
display(fg)
plotlib.save(fg, "d2_probability")

### Surprise and the logarithm (slide: Surprise and the Logarithm; plot `d2_surprise`)

In [ ]:
# slide: Surprise and the Logarithm
slide("Surprise and the Logarithm")
print(f"guess a number from 1 to 8: log2(8) = {math.log2(8):.0f} yes/no questions (2^3 = {2**3})")
pts = np.array([1, 1 / 2, 1 / 4, 1 / 8, 0.5, 0.25, 0.2, 0.05])
labels = ["certain", "1/2", "1/4", "1/8", "sun", "cloud", "rain", "snow"]
show(pl.DataFrame({"event": labels, "p": pts, "h = -log2 p (bits)": np.log2(1 / pts)}))

In [ ]:
# plot d2_surprise
fg, ax = plotlib.fig("full", 1.6)
pg = np.linspace(0.03, 1, 300)
ax.plot(pg, -np.log2(pg), color=BLUE)
for pv, lab, dx, dy in [(1, "$p=1$: 0 bits", -4, 8), (0.5, "$p=1/2$: 1 bit", 6, 4), (0.25, "$p=1/4$: 2 bits", 6, 4),
                        (0.125, "$p=1/8$: 3 bits", 6, 2), (0.05, "snow $p=0.05$: 4.32 bits", 6, 0)]:  # fmt: skip
    ax.scatter([pv], [-np.log2(pv)], color=RED, s=14, zorder=4)
    ax.annotate(
        lab, (pv, -np.log2(pv)), (dx, dy), textcoords="offset points", fontsize=7.5, ha="right" if dx < 0 else "left"
    )
ax.set(xlabel="probability $p(x)$ of the outcome", ylabel="surprise $h(x)$ (bits)", xlim=(0, 1.05), ylim=(-0.2, 5.3))
display(fg)
plotlib.save(fg, "d2_surprise")

### Two coins by hand (slide: Worked Example: Two Coins; plot `d2_entropy_coin`)

In [ ]:
# slide: Worked Example: Two Coins
slide("Worked Example: Two Coins")
for name, p_heads in [("fair coin", 0.5), ("biased coin", 0.9)]:
    p = np.array([p_heads, 1 - p_heads])
    h = -np.log2(p)
    show(pl.DataFrame({"outcome": ["heads", "tails"], "p": p, "h = -log2 p": h, "p * h": p * h}), digits=4)
    print(f"{name}: H = {p[0]:.1f} x {h[0]:.4f} + {p[1]:.1f} x {h[1]:.4f} = {np.sum(p * h):.4f} bits")
print("rounded as on the slide: -log2 0.9 = 0.152, -log2 0.1 = 3.32, 0.9 x 0.152 = 0.137, 0.1 x 3.32 = 0.332, H = 0.47")
print(f"exact products: 0.9 x h(heads) = {0.9 * -np.log2(0.9):.4f}, 0.1 x h(tails) = {0.1 * -np.log2(0.1):.4f}")
print(f"entropy of a certain outcome (p = 1): {entropy_bits([1.0]):.1f} bits")

### Entropy is the limit of compression (slide: Entropy Is the Limit of Compression; plot `d2_entropy_coin`)

100 000 tosses of a coin with $p(\text{heads}) = p$, one byte per toss. A compressor cannot beat $H(p)$ bits per toss on average.

In [ ]:
# slide: Worked Example: Two Coins; Entropy Is the Limit of Compression
slide("Entropy Is the Limit of Compression")
rng = np.random.default_rng(SEED)
N_TOSS = 100_000


rows = []
for p_heads in (0.5, 0.7, 0.9, 0.99):
    tosses = (rng.uniform(size=N_TOSS) < p_heads).astype(np.uint8) + 48  # ASCII '0' / '1'
    data = tosses.tobytes()
    rows.append({
        "p_heads": p_heads, "H_bits_per_toss": entropy_bits([p_heads, 1 - p_heads]),
        "zlib": bits_per_symbol(data, lambda b: zlib.compress(b, 9)), "bz2": bits_per_symbol(data, bz2.compress),
    })  # fmt: skip
coin_rows = pl.DataFrame(rows)
show(coin_rows)
print(
    "every compressor is above the entropy: the entropy is the limit, the gap is the price of not knowing p in advance"
)

In [ ]:
# plot d2_entropy_coin
fg, ax = plotlib.fig("full", 1.7)
ps = np.linspace(0.001, 0.999, 400)
ax.plot(ps, [entropy_bits([p, 1 - p]) for p in ps], color=BLUE, label="$H(p)$")
ax.scatter(coin_rows["p_heads"], coin_rows["zlib"], marker="s", s=14, color=ORANGE, zorder=4, label="zlib, measured")
ax.scatter(coin_rows["p_heads"], coin_rows["bz2"], marker="^", s=16, color=GREEN, zorder=4, label="bz2, measured")
ax.scatter([0.5, 0.9], [1.0, entropy_bits([0.9, 0.1])], color=RED, s=18, zorder=5)
ax.annotate("fair coin: 1 bit", (0.5, 1.0), (-6, 2), textcoords="offset points", fontsize=7.5, ha="right")
ax.annotate(
    "$p = 0.9$: 0.47 bits",
    (0.9, entropy_bits([0.9, 0.1])),
    (-8, -10),
    textcoords="offset points",
    fontsize=7.5,
    ha="right",
)
ax.set(xlabel="$p$ = probability of heads", ylabel="bits per toss", xlim=(0, 1), ylim=(0, 1.35))
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=3, borderaxespad=0.1, columnspacing=1.0, handletextpad=0.3
)
display(fg)
plotlib.save(fg, "d2_entropy_coin")

### Seven kinds of data (slides: Kolmogorov Complexity, To Learn Is to Compress; plot `d2_entropy_sources`)

Order-0 entropy sees only symbol frequencies. A compressor also sees repetition and structure.

In [ ]:
TEXT = (
    "Machine learning is the study of algorithms that improve their performance at some task through experience. "
    "A learning algorithm receives a finite set of examples and must produce a model that behaves well on examples "
    "it has never seen before. This is only possible when the examples share some regularity: a pattern that can be "
    "described more briefly than the examples themselves. If the data were pure noise, there would be nothing to "
    "learn, because no description shorter than the data itself would exist. The art of learning is therefore the "
    "art of finding short descriptions, of separating what is structure from what is accident. Simple models may "
    "miss part of the structure, while complex models may mistake accidents for structure. Both errors hurt the "
    "ability to predict new observations, and much of this course is about how to balance them. "
    "Compression and prediction are two sides of one coin. A model that predicts the next symbol well can be used "
    "to encode the data with few bits, and a code that uses few bits must have found the regularities of the data. "
    "The shortest program that prints a string is its Kolmogorov complexity, which no algorithm can compute in general, "
    "but a practical compressor gives an upper bound that is often good enough to compare two descriptions. "
    "When we choose between a straight line and a wiggly curve, we are really choosing between two descriptions of "
    "the same measurements: the short one with large errors and the long one with small errors. The description "
    "length of the model plus the description length of its errors is the number we want to keep small. "
    "A lookup table that stores every example has zero error, but its description is as long as the data. "
)


def entropy_bits_per_byte(data: bytes) -> float:
    counts = np.array(list(Counter(data).values()), dtype=float)
    return entropy_bits(counts / counts.sum())

In [ ]:
# slide: Kolmogorov Complexity; Entropy Is the Limit of Compression (plot d2_entropy_sources)
slide("Kolmogorov Complexity; Seven kinds of data")
rng = np.random.default_rng(SEED)
N = 4000
text = TEXT.encode()
shuffled = rng.permutation(np.frombuffer(text, dtype=np.uint8)).tobytes()
src_file = Path(textwrap.__file__).read_bytes()[:8000]
png_buf = BytesIO()  # a PNG image of Example 1, already compressed by the PNG format itself
fg, ax = plt.subplots(figsize=(4, 2))
ax.scatter(*example1()[:2], s=6)
fg.savefig(png_buf, format="png", dpi=100)
plt.close(fg)
sources = {
    "AB repeated": (b"AB" * N)[:N],
    "English text": text,
    "text, shuffled": shuffled,
    "text, 3 times": text * 3,
    "source file": src_file,
    "PNG image": png_buf.getvalue(),
    "random bytes": rng.bytes(N),
}
rows = []
for name, data in sources.items():
    rows.append({
        "source": name, "bytes": len(data), "H_bits_per_byte": entropy_bits_per_byte(data),
        "zlib": bits_per_symbol(data, lambda b: zlib.compress(b, 9)), "bz2": bits_per_symbol(data, bz2.compress),
    })  # fmt: skip
src_rows = pl.DataFrame(rows)
show(src_rows, 3)
program = 'print("AB" * 2000)'
print(
    f"AB repeated: {N} bytes, its shortest program here: {program!r} = {len(program)} characters; zlib: {len(zlib.compress(sources['AB repeated'], 9))} bytes"
)
print("random bytes: no program shorter than the data is known, zlib output is larger than the input:",
      len(zlib.compress(sources["random bytes"], 9)), "bytes for", N)  # fmt: skip

In [ ]:
# plot d2_entropy_sources
fg, ax = plotlib.fig("full", 1.9)
names = src_rows["source"].to_list()
xs, w = np.arange(len(names)), 0.27
for j, (col, colour, lab) in enumerate(
    [("H_bits_per_byte", BLUE, "entropy $H$"), ("zlib", ORANGE, "zlib"), ("bz2", GREEN, "bz2")]
):
    ax.bar(xs + (j - 1) * w, src_rows[col], width=w, color=colour, label=lab)
ax.set(xticks=xs, ylabel="bits per byte", ylim=(0, 9.5))
ax.set_xticklabels([s.replace(" ", "\n", 1) for s in names], fontsize=7)
ax.axhline(8, color=GREY, lw=0.6, ls=":")
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=3, borderaxespad=0.1, handletextpad=0.3, columnspacing=1.0
)
ax.grid(axis="x", visible=False)
display(fg)
plotlib.save(fg, "d2_entropy_sources")

**What to notice.** Random bytes sit at 8 bits for every measure and `zlib` adds overhead: no pattern, no compression.
Shuffling the English text keeps $H$ but destroys the order, and the compressors lose their advantage: the structure was in the order, not in the frequencies.
Repeating the text three times leaves $H$ unchanged and the compressors find the repetition. A PNG is already compressed.

## D3 — Model selection as compression (MDL)

Fit polynomials of degree $q = 0, \dots, 15$ to Example 1 ($n = 50$). A polynomial of degree $q$ has $m = q + 1$ parameters.
The two-part code length in bits (up to constants that do not depend on $q$; this is the BIC in bits):

$$\ell(M) + \ell(D \mid M) = \underbrace{\frac{m}{2}\log_2 n}_{\ell(M):\ \text{model}} + \underbrace{\frac{n}{2}\log_2 \text{MSE}_{\text{train}}}_{\ell(D \mid M):\ \text{residuals}}$$

Constants are dropped, so the values can be negative: only differences between models matter.

### The MDL table (slides: The MDL Trade-Off, Worked Example: Choosing a Degree; plots `d3_parts`, `d3_mdl_curve`)

In [ ]:
# slide: Worked Example: Choosing a Degree; The MDL Trade-Off
slide("Worked Example: Choosing a Degree; The MDL Trade-Off")


def mdl_table(x, y, x_test, y_test, max_q=15):
    """One row per degree q: model bits, error bits, total, train and test MSE."""
    n_pts = len(x)
    rows = []
    for q in range(min(max_q, n_pts - 1) + 1):
        p = Polynomial.fit(x, y, deg=q)
        tr, te = mse(y, p(x)), mse(y_test, p(x_test))
        model_bits = (q + 1) / 2 * math.log2(n_pts)
        error_bits = n_pts / 2 * math.log2(max(tr, 1e-300))
        rows.append({
            "q": q, "m": q + 1, "model_bits": model_bits, "error_bits": error_bits,
            "total_bits": model_bits + error_bits, "train_mse": tr, "test_mse": te,
        })  # fmt: skip
    return pl.DataFrame(rows)


x_train, y_train, x_test, y_test = example1()
n = len(x_train)
mdl = mdl_table(x_train, y_train, x_test, y_test)
show(mdl, 3)
q_mdl = int(mdl["q"][int(np.argmin(mdl["total_bits"].to_numpy()))])
q_oracle = int(mdl["q"][int(np.argmin(mdl["test_mse"].to_numpy()))])
print(f"log2({n}) = {math.log2(n):.4f}, so one parameter costs (1/2) log2 n = {math.log2(n) / 2:.4f} bits")
print(f"MDL choice (training data only): q = {q_mdl}, test MSE = {mdl['test_mse'][q_mdl]:.4f}")
print(f"lowest test error (oracle)     : q = {q_oracle}, test MSE = {mdl['test_mse'][q_oracle]:.4f}")
print(f"most complex model             : q = 15, test MSE = {mdl['test_mse'][15]:.4f}")
a, b = mdl.row(5, named=True), mdl.row(15, named=True)
print(
    f"q = 5 -> q = 15: errors save {a['error_bits'] - b['error_bits']:.1f} bits, the model costs {b['model_bits'] - a['model_bits']:.1f} bits more"
)
print(f"noise floor sigma^2 = {SIGMA**2:.2f}")

In [ ]:
# plot d3_parts (frame: The MDL Trade-Off): three degrees, model bits up, error bits down, total as a dot
fg, ax = plotlib.fig("full", 1.9)
degs = [1, q_mdl, 15]
names = ["too simple", "MDL choice", "too complex"]
sel = mdl.filter(pl.col("q").is_in(degs)).sort("q")
xs, w = np.arange(3), 0.34
ax.bar(xs - w / 2, sel["model_bits"], w, color=BLUE, label="model $\\ell(M)$")
ax.bar(xs + w / 2, sel["error_bits"], w, color=ORANGE, label="errors $\\ell(D \\mid M)$")
ax.scatter(xs, sel["total_bits"], marker="D", s=22, color=PURPLE, zorder=5, label="total")
for xi, row in zip(xs, sel.iter_rows(named=True)):
    ax.annotate(
        f"{row['model_bits']:.1f}",
        (xi - w / 2, row["model_bits"]),
        (0, 2),
        textcoords="offset points",
        ha="center",
        fontsize=7.5,
    )
    ax.annotate(
        f"{row['error_bits']:.1f}",
        (xi + w / 2, row["error_bits"]),
        (0, -2),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=7.5,
    )
ax.axhline(0, color="k", lw=0.6)
ax.set(xticks=xs, ylabel="bits", ylim=(-150, 62))
ax.set_xticklabels([f"{nm}, $q = {q}$\ntotal {t:.1f}" for nm, q, t in zip(names, degs, sel["total_bits"])])
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=3, borderaxespad=0.1, handletextpad=0.3, columnspacing=1.0
)
ax.grid(axis="x", visible=False)
display(fg)
plotlib.save(fg, "d3_parts")

In [ ]:
# plot d3_mdl_curve (frame: The MDL Trade-Off, The MDL Curve)
fg, ax = plotlib.fig("full", 1.8)
ax.plot(mdl["q"], mdl["model_bits"], color=BLUE, marker="o", ms=2.5, label="model $\\ell(M)$")
ax.plot(mdl["q"], mdl["error_bits"], color=ORANGE, marker="s", ms=2.5, label="errors $\\ell(D \\mid M)$")
ax.plot(mdl["q"], mdl["total_bits"], color=PURPLE, marker="D", ms=3, lw=1.8, label="total")
ax.axvline(q_mdl, color=RED, ls="--", lw=1)
ax.text(q_mdl + 0.3, -22, f"minimum: $q = {q_mdl}$", fontsize=7.5, color=RED)
ax.set(xlabel="polynomial degree $q$", ylabel="description length (bits)", xlim=(-0.3, 15.3))
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=3, borderaxespad=0.1, handletextpad=0.3, columnspacing=0.8
)
display(fg)
plotlib.save(fg, "d3_mdl_curve")

### Sweep: does the MDL choice pick the same degree every time? (no slide)

20 seeds per cell of the grid. `q_mdl` is the degree chosen from the training data only, `q_oracle` the degree with the lowest test error.
With $n = 15$ the degrees go up to 14 and the interpolating polynomial has training error near 0, so $\log_2 \text{MSE}$ goes to $-\infty$: the BIC approximation fails.

In [ ]:
rows = []
for n_s in (15, 50, 500):
    for sigma_s in (0.1, 0.3, 1.0):
        picks, te_mdl, te_best, te_full = [], [], [], []
        for s in range(20):
            xa, ya, xb, yb = example1(n_s, sigma_s, seed=SEED + s)
            t = mdl_table(xa, ya, xb, yb)
            q_pick = int(np.argmin(t["total_bits"].to_numpy()))
            picks.append(q_pick)
            te_mdl.append(t["test_mse"][q_pick])
            te_best.append(t["test_mse"].min())
            te_full.append(t["test_mse"][-1])
        picks = np.array(picks)
        rows.append({
            "n": n_s, "sigma": sigma_s, "q_mdl_median": float(np.median(picks)), "q_mdl_min": int(picks.min()),
            "q_mdl_max": int(picks.max()), "share_q_3_to_5": float(np.mean((picks >= 3) & (picks <= 5))),
            "test_mse_mdl": float(np.mean(te_mdl)), "test_mse_oracle": float(np.mean(te_best)),
            "test_mse_max_degree": float(np.median(te_full)),
        })  # fmt: skip
sweep3 = pl.DataFrame(rows)
show(sweep3, 3)

**Reading the sweep.** For $n = 50$ and $500$ the MDL choice sits at 3 to 5 in almost every run and its test error is within a few percent of the oracle:
it does not always pick 5, because the test curve is flat between 3 and 5. With a large noise ($\sigma = 1$) the data support only a simpler model and the choice moves down.
With $n = 15$ the approximation breaks: the interpolating polynomial (degree 14) has a training error near 0, so the error cost goes to $-\infty$, the BIC always picks it, and its test error explodes.
BIC is a large-sample approximation; with very few points use a validation estimate (D4).

**Takeaway.** Training error always decreases with complexity, so it cannot choose the model.
The description length stops decreasing once the extra parameters only encode noise, and MDL picks a model whose test error is close to the best one **without looking at test data**.

## Bridge — finding the model is an optimization problem

`Polynomial.fit` solved for the parameters in closed form. In general there is no closed form: we must **search** the parameter space for the lowest loss.
The loss surface of the simplest model, $\hat{y} = w\,x + b$, on Example 1 (slides: Finding the Model Is an Optimization Problem, Why Not Try Every Value; plot `d3_loss_surface`).

In [ ]:
# slide: Finding the Model Is an Optimization Problem; Why Not Try Every Value?
slide("Finding the Model Is an Optimization Problem; Why Not Try Every Value?")
x_train, y_train, _, _ = example1()
w_grid, b_grid = np.meshgrid(np.linspace(-6, 4, 200), np.linspace(-2, 3, 200))
loss = np.mean((w_grid[..., None] * x_train + b_grid[..., None] - y_train) ** 2, axis=-1)
i, j = np.unravel_index(loss.argmin(), loss.shape)
w_ls, b_ls = np.polyfit(x_train, y_train, 1)
print(
    f"grid search: 200 x 200 = {loss.size} loss evaluations, minimum at w = {w_grid[i, j]:.3f}, b = {b_grid[i, j]:.3f}, MSE = {loss[i, j]:.4f}"
)
print(
    f"closed form (least squares):        w = {w_ls:.3f}, b = {b_ls:.3f}, MSE = {mse(y_train, w_ls * x_train + b_ls):.4f}"
)
for params in (1, 2, 3, 5, 10, 1000):
    print(
        f"{params:5d} parameters, 200 values each: 10^{params * math.log10(200):.1f} = {10 ** (params * math.log10(200) % 1):.3g}e{int(params * math.log10(200))} loss evaluations"
    )

In [ ]:
# plot d3_loss_surface
fg, ax = plotlib.fig("col", 2.0)
cs = ax.contourf(w_grid, b_grid, np.log10(loss), levels=14, cmap="Blues_r")
ax.scatter([w_grid[i, j]], [b_grid[i, j]], marker="*", s=70, color=RED, zorder=5)
ax.annotate("minimum", (w_grid[i, j], b_grid[i, j]), (8, 8), textcoords="offset points", fontsize=7.5, color=RED,
            bbox={"fc": "white", "ec": "none", "pad": 0.6, "alpha": 0.85})  # fmt: skip
ax.set(xlabel="$w$ (slope)", ylabel="$b$ (intercept)")
ax.grid(False)
cb = fg.colorbar(cs, ax=ax, pad=0.03, aspect=18)
cb.set_label("$\\log_{10}$ MSE", fontsize=7.5)
cb.ax.tick_params(labelsize=7)
display(fg)
plotlib.save(fg, "d3_loss_surface")

A 200×200 grid needs **40 000** loss evaluations for 2 parameters, $200^{10} \approx 10^{23}$ for 10, and the number for 1 000 parameters is not worth printing.
That is why Class 02 covers **blind (population-based)** and **gradient-based** optimization.